In [0]:
# ─────────────────────────────────────────────────────────────
# run_framework.py  |  Medallion ETL  |  L0 / L1 / L2
# ─────────────────────────────────────────────────────────────

In [0]:
# WIDGETS

dbutils.widgets.text("GROUP_ID",          "")
dbutils.widgets.text("TARGET_LOAD_TABLE", "")
dbutils.widgets.text("ENVIRONMENT",       "dev")

GID   = dbutils.widgets.get("GROUP_ID").strip().upper()
TBL   = dbutils.widgets.get("TARGET_LOAD_TABLE").strip()
ENV   = dbutils.widgets.get("ENVIRONMENT").strip() or "dev"
LAYER = "L0" if GID.endswith("_L0") else "L1" if GID.endswith("_L1") else "L2" if GID.endswith("_L2") else "ALL"

if not GID:
    raise Exception("GROUP_ID is required. Set the widget or job parameter.")

print(f"GROUP_ID : {GID}  |  LAYER : {LAYER}  |  TABLE FILTER : {TBL or 'ALL'}  |  ENV : {ENV}")

In [0]:
# CATALOG

try:
    _cats   = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    CATALOG = "demo_catalog"   if "demo_catalog"   in _cats else \
              "hive_metastore" if "hive_metastore" in _cats else _cats[0]
except Exception:
    CATALOG = "hive_metastore"

print(f"CATALOG  : {CATALOG}")

In [0]:
# IMPORTS

import io, traceback, requests, pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
# READ SOURCE
# Supports: csv, json, parquet, excel  →  HTTP / HTTPS URLs (pandas)
#           csv, json, parquet, delta  →  cloud paths  dbfs / s3 / abfss

def read_source(url, fmt, delim=","):
    fmt = (fmt or "csv").strip().lower()
    url = (url or "").strip()
    if not url:
        raise ValueError("SOURCE_URL is empty in control table.")

    if url.lower().startswith("http"):
        # ── HTTP source — download via requests then parse ────
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw = io.BytesIO(r.content)
        if   fmt == "csv":              return spark.createDataFrame(pd.read_csv(raw, sep=delim or ","))
        elif fmt == "json":             return spark.createDataFrame(pd.read_json(raw))
        elif fmt == "parquet":          return spark.createDataFrame(pd.read_parquet(raw))
        elif fmt in ("excel","xlsx"):   return spark.createDataFrame(pd.read_excel(raw))
        else: raise ValueError(f"HTTP source does not support format '{fmt}'. Use csv/json/parquet/excel.")
    else:
        # ── Cloud path — Spark native ─────────────────────────
        if   fmt == "csv":     return spark.read.option("header","true").option("inferSchema","true").option("sep", delim or ",").csv(url)
        elif fmt == "json":    return spark.read.option("multiLine","true").json(url)
        elif fmt == "parquet": return spark.read.parquet(url)
        elif fmt == "delta":   return spark.read.format("delta").load(url)    # ← Delta table path
        else: raise ValueError(f"Cloud source does not support format '{fmt}'. Use csv/json/parquet/delta.")

In [0]:
# WRITE TABLE  (FULL / INCREMENTAL / MERGE)

def write_table(df, schema, table, load_type, merge_keys=""):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    full = f"{CATALOG}.{schema}.{table}"
    lt   = (load_type or "FULL").strip().upper()

    if lt == "FULL":
        df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full)

    elif lt in ("INCREMENTAL", "APPEND"):
        df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(full)

    elif lt == "MERGE":
        if not merge_keys:
            raise ValueError(f"MERGE_KEYS is empty for {full}. Add keys to control table.")
        keys  = [k.strip() for k in merge_keys.split(",")]
        on    = " AND ".join([f"t.{k}=s.{k}" for k in keys])
        upd   = ", ".join([f"t.{c}=s.{c}" for c in df.columns if c not in keys])
        ins_c = ", ".join(df.columns)
        ins_v = ", ".join([f"s.{c}" for c in df.columns])
        tmp   = f"_fw_{table}"
        df.createOrReplaceTempView(tmp)
        spark.sql(f"CREATE TABLE IF NOT EXISTS {full} USING DELTA AS SELECT * FROM {tmp} WHERE 1=0")
        spark.sql(f"""
            MERGE INTO {full} t USING {tmp} s ON {on}
            WHEN MATCHED     THEN UPDATE SET {upd}
            WHEN NOT MATCHED THEN INSERT ({ins_c}) VALUES ({ins_v})
        """)
    elif lt == "DELTA":
        # DELTA = MERGE into target using merge_keys
        # If no merge_keys → overwrite (fallback)
        if merge_keys:
            keys  = [k.strip() for k in merge_keys.split(",")]
            on    = " AND ".join([f"t.{k}=s.{k}" for k in keys])
            upd   = ", ".join([f"t.{c}=s.{c}" for c in df.columns if c not in keys])
            ic    = ", ".join(df.columns)
            iv    = ", ".join([f"s.{c}" for c in df.columns])
            tmp   = f"_fw_d_{table}"
            df.createOrReplaceTempView(tmp)
            spark.sql(f"CREATE TABLE IF NOT EXISTS {full} USING DELTA AS SELECT * FROM {tmp} WHERE 1=0")
            spark.sql(f"MERGE INTO {full} t USING {tmp} s ON {on} WHEN MATCHED THEN UPDATE SET {upd} WHEN NOT MATCHED THEN INSERT ({ic}) VALUES ({iv})")
        else:
            df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full)

    else:
        print(f"  Unknown LOAD_TYPE '{lt}' — using FULL")
        df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full)

    return spark.table(full).count()

In [0]:
# AUDIT LOG

def audit(table, layer, status, rows, msg, t0, t1):
    try:
        spark.sql(f"""
            INSERT INTO {CATALOG}.admin.audit_log VALUES (
                '{GID}','{table}','{layer}','{status}',
                {rows},'{str(msg).replace("'","''")[:400]}',
                '{t0:%Y-%m-%d %H:%M:%S}','{t1:%Y-%m-%d %H:%M:%S}',
                current_timestamp())
        """)
    except Exception as e:
        print(f"  audit write failed: {e}")

In [0]:
# RESOLVE CATALOG  — add catalog prefix to bare schema refs in SQL

def resolve(sql, src_schema, src_table):
    q = sql
    if src_schema and f"{src_schema}." in q and f"{CATALOG}.{src_schema}." not in q:
        q = q.replace(f"{src_schema}.", f"{CATALOG}.{src_schema}.")
    if src_table and src_schema:
        for kw in ("FROM ", "JOIN "):
            if f"{kw}{src_table}" in q and f"{kw}{CATALOG}." not in q:
                q = q.replace(f"{kw}{src_table}", f"{kw}{CATALOG}.{src_schema}.{src_table}")
    return q

In [0]:
# PROCESS ONE DETAIL TABLE

def run_layer(detail_table, layer):
    print(f"\n{'='*55}\n  {layer} — {detail_table}\n{'='*55}")

    tbl_filter = f"AND TARGET_TABLE = '{TBL}'" if TBL else ""

    rows = spark.sql(f"""
        SELECT * FROM {CATALOG}.admin.{detail_table}
        WHERE DATA_FLOW_GROUP_ID = '{GID}'
        AND   IS_ACTIVE          = 'Y'
        AND   UPPER(LOAD_TYPE)  != 'VIEW'
        {tbl_filter}
        ORDER BY TARGET_TABLE
    """).collect()

    if not rows:
        print(f"  No rows found — skipping {layer}.")
        return

    print(f"  Tables : {len(rows)}")
    ok = True

    for row in rows:
        r          = row.asDict()
        tgt_sch    = (r.get("TARGET_SCHEMA")       or "").strip()
        tgt_tbl    = (r.get("TARGET_TABLE")        or "").strip()
        load_type  = (r.get("LOAD_TYPE")           or "FULL").strip()
        merge_keys = (r.get("MERGE_KEYS")          or "").strip()
        src_url    = (r.get("SOURCE_URL")          or "").strip()
        src_sch    = (r.get("SOURCE_OBJ_SCHEMA")   or "").strip()
        src_tbl    = (r.get("SOURCE_OBJ_NAME")     or "").strip()
        file_fmt   = (r.get("INPUT_FILE_FORMAT") or r.get("FILE_FORMAT") or "csv").strip()
        delim      = (r.get("DELIMETER")           or ",").strip()
        trans_q    = (r.get("TRANSFORMATION_QUERY") or "").strip()

        if not tgt_tbl or not tgt_sch:
            print("  ⚠ Skipping — TARGET_TABLE or TARGET_SCHEMA empty"); continue

        full = f"{CATALOG}.{tgt_sch}.{tgt_tbl}"
        print(f"\n  ▶  {full}  |  {load_type}")

        t0 = datetime.now(); status = "FAILED"; n = 0; msg = ""

        try:
            # ── READ ──────────────────────────────────────────
            lt_up = load_type.strip().upper()

            if layer == "L0":
                if lt_up == "DELTA" and not src_url and src_sch and src_tbl:
                    # LOAD_TYPE=DELTA with no SOURCE_URL — read from existing Delta table
                    df = spark.table(f"{CATALOG}.{src_sch}.{src_tbl}")
                    print(f"     Source : {CATALOG}.{src_sch}.{src_tbl} (delta table)")
                else:
                    df = read_source(src_url, file_fmt, delim)
            else:
                if trans_q:
                    df = spark.sql(resolve(trans_q, src_sch, src_tbl))
                else:
                    df = spark.table(f"{CATALOG}.{src_sch}.{src_tbl}")
                    print(f"     Source : {CATALOG}.{src_sch}.{src_tbl}")

            # ── ETL COLUMNS ───────────────────────────────────
            df = df.withColumn("_etl_group_id", F.lit(GID)) \
                   .withColumn("_etl_layer",    F.lit(layer)) \
                   .withColumn("_etl_env",      F.lit(ENV)) \
                   .withColumn("_etl_load_ts",  F.current_timestamp())

            # ── WRITE ─────────────────────────────────────────
            n      = write_table(df, tgt_sch, tgt_tbl, load_type, merge_keys)
            status = "SUCCESS"
            msg    = f"{n:,} rows"
            print(f"     ✅  {msg}  →  {full}")

        except Exception as e:
            ok  = False
            msg = f"{type(e).__name__}: {str(e)[:300]}"
            print(f"     ❌  {msg}")

        finally:
            t1 = datetime.now()
            print(f"     ⏱  {round((t1-t0).total_seconds(),1)}s")
            audit(tgt_tbl, layer, status, n, msg, t0, t1)

    if not ok:
        raise Exception(
            f"{layer} had failures — GROUP_ID='{GID}'\n"
            f"Check: SELECT * FROM {CATALOG}.admin.audit_log "
            f"WHERE DATA_FLOW_GROUP_ID='{GID}' ORDER BY LOAD_TS DESC"
        )

In [0]:
# EXECUTE

t_start = datetime.now()

if   LAYER == "L0": run_layer("data_flow_l0_detail", "L0")
elif LAYER == "L1": run_layer("data_flow_l1_detail", "L1")
elif LAYER == "L2": run_layer("data_flow_l2_detail", "L2")
else:
    run_layer("data_flow_l0_detail", "L0")
    run_layer("data_flow_l1_detail", "L1")
    run_layer("data_flow_l2_detail", "L2")

print(f"\n{'='*55}")
print(f"  DONE  |  {GID}  |  {round((datetime.now()-t_start).total_seconds(),1)}s")
print(f"{'='*55}")